# 🏠 Real Estate Intelligence Platform
## Notebook 04 — Model Training & Comparison

**Project:** AI-Powered Real Estate Intelligence, Valuation & Investment Recommendation Platform  
**Phase:** Phase 6 (Machine Learning Model Development)  
**Input:** data/model_df.csv (tree models) · data/model_df_scaled.csv (linear regression)  
**Target:** Estimated_Price  

---

### Models to Train
| # | Model | Dataset |
|---|---|---|
| 1 | Linear Regression | Scaled |
| 2 | Random Forest | Unscaled |
| 3 | XGBoost | Unscaled |
| 4 | CatBoost | Unscaled |
| 5 | LightGBM | Unscaled |
| 6 | Gradient Boosting | Unscaled |

### Evaluation Metrics
MAE · RMSE · MSE · R² Score

In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Load data
df = pd.read_csv('../data/processed/model_df.csv')
df_scaled = pd.read_csv('../data/processed/model_df_scaled.csv')

# Load feature columns
with open('../models/feature_columns.json', 'r') as f:
    FEATURES = json.load(f)

TARGET = 'Estimated_Price'

# Split unscaled
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split scaled (linear regression only)
X_scaled = df_scaled[FEATURES]
y_scaled = df_scaled[TARGET]
X_train_sc, X_test_sc, y_train_sc, y_test_sc = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Features:", len(FEATURES))
print("Target:", TARGET)

Train size: (200000, 34)
Test size: (50000, 34)
Features: 34
Target: Estimated_Price


In [4]:
#  Evaluation helper function:
results = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, fit=True):
    if fit:
        model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_te, y_pred)
    results[name] = {'MAE': round(mae,2), 'MSE': round(mse,2),
                     'RMSE': round(rmse,2), 'R2': round(r2,4)}
    print(f"\n{name}")
    print(f"  MAE:  {mae:.2f}")
    print(f"  MSE:  {mse:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.4f}")
    return model

In [6]:
# Linear Regression:
lr = LinearRegression()
lr = evaluate_model('Linear Regression', lr, 
                     X_train_sc, y_train_sc, 
                     X_test_sc, y_test_sc)


Linear Regression
  MAE:  31.64
  MSE:  2010.92
  RMSE: 44.84
  R²:   0.8569


In [7]:
# Random Forest:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf = evaluate_model('Random Forest', rf, X_train, y_train, X_test, y_test)


Random Forest
  MAE:  13.29
  MSE:  331.01
  RMSE: 18.19
  R²:   0.9764


In [8]:
# XGBoost:
xgb = XGBRegressor(n_estimators=100, random_state=42, 
                    learning_rate=0.1, n_jobs=-1, verbosity=0)
xgb = evaluate_model('XGBoost', xgb, X_train, y_train, X_test, y_test)


XGBoost
  MAE:  13.68
  MSE:  325.00
  RMSE: 18.03
  R²:   0.9769


In [9]:
# Catboost:
cat = CatBoostRegressor(iterations=100, random_seed=42, 
                         learning_rate=0.1, verbose=0)
cat = evaluate_model('CatBoost', cat, X_train, y_train, X_test, y_test)


CatBoost
  MAE:  15.30
  MSE:  412.66
  RMSE: 20.31
  R²:   0.9706


In [10]:
# LightGBM:
lgbm = LGBMRegressor(n_estimators=100, random_state=42, 
                      learning_rate=0.1, n_jobs=-1, verbose=-1)
lgbm = evaluate_model('LightGBM', lgbm, X_train, y_train, X_test, y_test)


LightGBM
  MAE:  13.75
  MSE:  328.03
  RMSE: 18.11
  R²:   0.9767


In [11]:
# Gradient Boosting:
gb = GradientBoostingRegressor(n_estimators=100, random_state=42, 
                                learning_rate=0.1)
gb = evaluate_model('Gradient Boosting', gb, X_train, y_train, X_test, y_test)


Gradient Boosting
  MAE:  19.00
  MSE:  688.60
  RMSE: 26.24
  R²:   0.9510


In [12]:
import plotly.express as px

# Comparison table
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print("Model Comparison (sorted by R²):")
print(results_df.to_string())

# Plot
fig = px.bar(results_df.reset_index(), x='index', y='R2',
             title='Model R² Comparison',
             labels={'index': 'Model', 'R2': 'R² Score'},
             color='R2', color_continuous_scale='Greens')
fig.update_layout(xaxis_tickangle=-20)
fig.show()


Model Comparison (sorted by R²):
                     MAE      MSE   RMSE      R2
XGBoost            13.68   325.00  18.03  0.9769
LightGBM           13.75   328.03  18.11  0.9767
Random Forest      13.29   331.01  18.19  0.9764
CatBoost           15.30   412.66  20.31  0.9706
Gradient Boosting  19.00   688.60  26.24  0.9510
Linear Regression  31.64  2010.92  44.84  0.8569


In [13]:
# Check train vs test R² for XGBoost
train_pred = xgb.predict(X_train)
test_pred = xgb.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"XGBoost Train R²: {train_r2:.4f}")
print(f"XGBoost Test  R²: {test_r2:.4f}")
print(f"Difference: {(train_r2 - test_r2):.4f}")
print("Verdict:", "Overfitting" if (train_r2 - test_r2) > 0.05 else "Healthy fit")

XGBoost Train R²: 0.9784
XGBoost Test  R²: 0.9769
Difference: 0.0015
Verdict: Healthy fit


In [14]:
# Best model = XGBoost (highest R², lowest RMSE)
best_model = xgb
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(lr, '../models/linear_regression.pkl')  # save LR separately
print("\nBest model: XGBoost (R²=0.9769)")
print("Saved best_model.pkl + linear_regression.pkl")


Best model: XGBoost (R²=0.9769)
Saved best_model.pkl + linear_regression.pkl


---

## ✅ Notebook 04 Complete

### Model Comparison Results
| Model | MAE | MSE | RMSE | R² |
|---|---|---|---|---|
| XGBoost | 13.68 | 325.00 | 18.03 | 0.9769 |
| LightGBM | 13.75 | 328.03 | 18.11 | 0.9767 |
| Random Forest | 13.29 | 331.01 | 18.19 | 0.9764 |
| CatBoost | 15.30 | 412.66 | 20.31 | 0.9706 |
| Gradient Boosting | 19.00 | 688.60 | 26.24 | 0.9510 |
| Linear Regression | 31.64 | 2010.92 | 44.84 | 0.8569 |

### Best Model — XGBoost
| Check | Value | Verdict |
|---|---|---|
| Train R² | 0.9784 | — |
| Test R² | 0.9769 | — |
| Difference | 0.0015 | ✅ Healthy fit, no overfitting |

### Key Observations
- XGBoost and LightGBM virtually identical (R² diff = 0.0002) — both excellent
- Random Forest competitive but slower to train
- Gradient Boosting weakest among tree models — slowest + highest RMSE
- Linear Regression baseline R² 0.857 — confirms non-linear relationships exist
- All tree models outperform Linear Regression significantly

### Saved Models
| File | Description |
|---|---|
| models/best_model.pkl | XGBoost — primary model for Streamlit |
| models/linear_regression.pkl | Linear Regression — scaled input required |

### Next Step
→ Open `05_explainability.ipynb` for SHAP Global + Local Explainability